# Schema-Aware NL2SQL — Fine-Tune on Spider (one notebook)

Runs the whole pipeline on a **free Colab T4 GPU**: clone → install → get Spider data → smoke test → LoRA fine-tune T5 → execution-accuracy eval → push merged model to the Hugging Face Hub.

**Before you start:**
1. `Runtime → Change runtime type → T4 GPU`.
2. Have a **Hugging Face write token** ready (huggingface.co/settings/tokens).
3. Download **`spider.zip`** once from the official Spider release (Yale LILY / Spider 1.0). You'll upload it in Step 3 (or set a Google Drive file id to auto-download).

## Step 0 — Confirm the GPU

In [ ]:
!nvidia-smi

## Step 1 — Settings (edit these)

In [ ]:
REPO_BRANCH = "feature/schema-aware-anydb"
BASE_MODEL  = "t5-base"          # use "t5-large" for best accuracy (slower, more VRAM)
EPOCHS      = 5
MODEL_REPO  = "Srijan-Ratrey/nl2sql-t5-spider"   # where the merged model gets pushed

# Optional: Google Drive file id for spider.zip to auto-download.
# Leave "" to upload spider.zip manually in Step 3.
SPIDER_GDRIVE_ID = ""
print("base:", BASE_MODEL, "| epochs:", EPOCHS, "| push to:", MODEL_REPO)

## Step 2 — Clone the repo & install dependencies

In [ ]:
# Always start from /content and remove any previous clone, so re-running this
# cell never creates a nested copy.
%cd /content
!rm -rf Schema-Aware-Natural-Language-to-SQL-Agent
!git clone -b {REPO_BRANCH} https://github.com/Srijan-Ratrey/Schema-Aware-Natural-Language-to-SQL-Agent.git
%cd /content/Schema-Aware-Natural-Language-to-SQL-Agent
!pip -q install transformers datasets peft accelerate evaluate sqlglot sentencepiece
# Colab preinstalls an old torchao (0.10.0) that recent peft rejects and errors on
# during get_peft_model. We don't use torchao, so remove it to avoid the ImportError.
!pip -q uninstall -y torchao
# Spider's HF loader may require this flag on newer `datasets` versions:
%env HF_DATASETS_TRUST_REMOTE_CODE=1

# Authenticate to the HF Hub from Colab Secrets (left panel: add a secret named HF_TOKEN
# with a WRITE token and enable "Notebook access"). This silences the unauthenticated
# warning, speeds downloads, and lets the push step (Step 7) run without a prompt.
import os
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    from huggingface_hub import login
    login(token=os.environ["HF_TOKEN"])
    print("HF auth OK")
except Exception as e:
    print("HF auth skipped (set an HF_TOKEN secret to enable):", e)

## Step 3 — Get the Spider data (schemas + databases)

Needs `tables.json` (schemas) and `database/` (the SQLite DBs, for execution-accuracy eval). Upload `spider.zip` when prompted, or set `SPIDER_GDRIVE_ID` above to auto-download.

In [ ]:
import os, glob, zipfile

if SPIDER_GDRIVE_ID:
    !pip -q install gdown
    !gdown {SPIDER_GDRIVE_ID} -O /content/spider.zip
else:
    from google.colab import files
    print("Upload spider.zip (downloaded from the official Spider release)...")
    files.upload()

# Extract any uploaded spider*.zip (from /content or the cwd) into /content/spider_data
for z in glob.glob("/content/spider*.zip") + glob.glob("spider*.zip"):
    print("Extracting", z)
    with zipfile.ZipFile(z) as f:
        f.extractall("/content/spider_data")

# Resolve ABSOLUTE paths so later cells work regardless of the current directory.
_tables = glob.glob("/content/**/tables.json", recursive=True)
TABLES_JSON = os.path.abspath(_tables[0]) if _tables else None
_db_dirs = [d for d in glob.glob("/content/**/database", recursive=True) if os.path.isdir(d)]
DB_DIR = os.path.abspath(_db_dirs[0]) if _db_dirs else None
print("TABLES_JSON:", TABLES_JSON)
print("DB_DIR:", DB_DIR)
assert TABLES_JSON and DB_DIR, "Could not find tables.json / database/ inside the zip."

## Step 4 — Smoke test (~2 min)
Validates the whole pipeline on a tiny sample before committing hours of compute.

**Expect ~0% accuracy here — that is normal.** 200 samples / 1 epoch is far too little to
teach the model SQL; this step only confirms that train → save → eval runs without errors.
Real accuracy comes from the full run in Step 5.

In [ ]:
!python scripts/train_lora.py \
  --base-model t5-base \
  --tables-json "{TABLES_JSON}" \
  --epochs 1 --max-train-samples 200 \
  --output-dir smoke-test
# Scroll up: the smoke test passed only if there is NO traceback above and you see
# training logs ending with a save to 'smoke-test/'.

## Step 5 — Full fine-tune
Saves the LoRA adapter to `nl2sql-t5-lora/` (checkpoint each epoch). ~2–4h on a T4 for `t5-large`. If you hit OOM, re-run with `--batch-size 2 --grad-accum 16` or use `t5-base`.

**What's tuned for accuracy & speed:** LoRA now adapts the `q,k,v,o` attention projections (wider than q/v alone), training uses a cosine LR schedule with warmup, and **early stopping** keeps the best-generalizing checkpoint — so it may stop before `EPOCHS` and that's expected, not a failure. Per-epoch eval reports **loss only** (fast); the real execution accuracy comes from Step 6.

Extra knobs (optional): `--lora-targets q,k,v,o,wi,wo` also adapts the feed-forward layers (higher accuracy ceiling, more VRAM/time); `--patience 0` disables early stopping.

> **On a MacBook (Apple Silicon / M4 Pro)?** Don't use this Colab — run `scripts/train_lora.py` locally instead. See the "Train locally on Apple Silicon" section in `docs/NL2SQL_WALKTHROUGH.md` (use `t5-base`, fp32 on MPS, `PYTORCH_ENABLE_MPS_FALLBACK=1`).

In [ ]:
!python scripts/train_lora.py \
  --base-model "{BASE_MODEL}" \
  --tables-json "{TABLES_JSON}" \
  --epochs {EPOCHS} \
  --output-dir nl2sql-t5-lora

## Step 6 — Execution-accuracy evaluation
Runs predicted vs. gold SQL against the real DBs and writes the number + samples to `docs/EVAL.md`. Drop `--limit` for the full dev set (slower).

In [ ]:
!python scripts/evaluate_spider.py \
  --base-model "{BASE_MODEL}" --adapter ./nl2sql-t5-lora \
  --tables-json "{TABLES_JSON}" \
  --spider-db-dir "{DB_DIR}" \
  --limit 200

print("\n===== docs/EVAL.md =====")
print(open("docs/EVAL.md").read())

## Step 7 — Push the merged model to the Hugging Face Hub
Merges the saved adapter into the base (no retraining) and uploads. Uses the `HF_TOKEN`
secret you set in Step 2 — no prompt. The token must have **write** scope.

In [ ]:
# HF auth was set from Colab Secrets in Step 2, so no token prompt is needed here.
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from peft import PeftModel

tok = AutoTokenizer.from_pretrained(BASE_MODEL)
base = AutoModelForSeq2SeqLM.from_pretrained(BASE_MODEL)
merged = PeftModel.from_pretrained(base, "/content/nl2sql-t5-lora").merge_and_unload()
merged.push_to_hub(MODEL_REPO)
tok.push_to_hub(MODEL_REPO)
print("\nPushed merged model to:", MODEL_REPO)

## Done 🎉

Next:
- Point serving at your model: set the default in `src/nl2sql_agent.py` / `config.py` and the HF Space `MODEL_ID` to **`MODEL_REPO`**.
- Commit the generated `docs/EVAL.md` and update your README/resume bullet with the **measured** execution accuracy.

Tip: download the trained adapter so you don't lose it when the runtime recycles:
```python
from google.colab import files; import shutil
shutil.make_archive('nl2sql-t5-lora', 'zip', 'nl2sql-t5-lora'); files.download('nl2sql-t5-lora.zip')
```